# Structural Training — EfficientNet-B0 · 3-class CNN
## clean / adversarial / tampered

This is the report-facing Structural Training phase. It consolidates FYP1's two image-level detectors into **one EfficientNet-B0, 3-class** model for real-time deployment. The tampered class uses physical-manipulation patterns such as sticker overlay, occlusion, finder-pattern damage and scratches. Blur, glare, perspective and camera noise are capture conditions, not attack labels.

Output for the Risk Decision Layer: `p_structural = 1 − P(clean)`. Output for the UI: predicted structural type.

| Phase | What |
|---|---|
| 0 | Setup, Drive, seeds, paths |
| 1 | Build the clean QR base set |
| 2 | Split base groups first, then derive adversarial/tampered variants |
| 3 | Fine-tune EfficientNet-B0 |
| 4 | Evaluate aggregate, per-class, camera and source slices |
| 5 | Temperature calibration |
| 6 | Structural probability behaviour and deployment gates |
| 7 | ONNX export, parity and latency |

**Before running:** Runtime → Change runtime type → **T4 GPU**, then Run all. The deployed baseline is never overwritten by this notebook.


In [ ]:
# ============ Phase 0 - Setup ============
# ORDER MATTERS HERE. Mount Drive BEFORE installing anything.
#
# torchattacks declares requests~=2.25.1. Installed normally it downgrades the
# requests that google.colab itself depends on (it pins 2.32.4), and the next
# drive.mount() then fails with "ValueError: mount failed". Its real
# dependencies - torch, torchvision, scipy, tqdm, numpy - all ship with Colab
# already, so --no-deps installs it without touching requests.
#
# optimum[onnxruntime] was dropped: it was never imported, and it drags in
# transformers/datasets, which demand a newer requests and widen the conflict.
#
# ALREADY HIT THAT ERROR? The broken requests is still on the VM. Use
# Runtime -> Disconnect and delete runtime, reopen, then run this cell.
# A plain "Restart session" keeps the bad package and is not enough.

from google.colab import drive
drive.mount('/content/drive')

%pip -q install qrcode[pil] onnx onnxruntime onnxscript scikit-learn
%pip -q install --no-deps torchattacks
# onnxruntime installs its own sympy, which can be older than the one torch
# needs. torch then fails on `from torchvision import models` in Phase 2 with
# "module 'sympy' has no attribute 'printing'", raised fourteen frames deep in
# torch._dynamo - so it reads as a torchvision problem rather than a dependency
# one. Restoring sympy after the ONNX packages keeps both halves working.
%pip -q install -q -U sympy

# Prove the combination actually imports, here, rather than in Phase 2 after
# Phase 1 has spent minutes generating codes. If the kernel is still holding the
# packages it loaded before the installs, this is where you find out.
try:
    import sympy.printing  # noqa: F401
    from torchvision import models as _models  # noqa: F401
except Exception as _exc:
    raise SystemExit(
        f"Dependency clash after install: {_exc}\n\n"
        "The packages on disk are fine; this kernel is holding stale ones.\n"
        "Runtime -> Restart session, then run this cell again.\n"
        "Phase 1 and Phase 2 are checkpointed, so nothing already built is lost."
    )

import os, json, random, time, math, hashlib, csv
import numpy as np
import torch, torch.nn as nn
from PIL import Image

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

BASE = '/content/drive/MyDrive/FYP2/structural'

# RUN 3. The lineage, all measured on the real emulator camera:
#   RUN 1  blur was one of the synthetic attacks, so the model learned
#          "blurry = tampered" and scored a real photo of a SAFE code 0.9995.
#   RUN 2  dropped blur, camera-simulated every class. Its own simulated
#          captures fell to 0.06, but the real crop only reached 0.5973 - still
#          the wrong side of 0.5.
#   RUN 3  names the remaining gap. RUN 2 warped the code alone and replicated
#          its edge pixels, so the model never saw a code with a wall or a table
#          AROUND it, while a real crop always has some. Compositing the code
#          onto real room texture reproduced the failure exactly: mean 0.6089
#          against the real crop's 0.5973, versus 0.0627 without background.
#          So RUN 3 puts every code on a surface before photographing it.
#   RUN 4  RUN 3 works - the real crop reads 0.0011 clean - but its clean
#          class was 2500 plain black-on-white codes, so 'pristine clean
#          recall' was 1.0000 on a question nobody struggles with. RUN 4 gives
#          clean codes the variety real ones have, above all a CENTRED LOGO,
#          which looks like a sticker and is not one. It also spreads the
#          attacks over a severity range instead of making them all large.
#          The numbers drop because the question got harder, not because the
#          model got worse.
#   RUN 5  RUN 4's logo fix held - 0 of 20 branded codes flagged against RUN 3's
#          20 of 20 - but it cost the margin on real captures. Five genuine
#          frames from the app scored 0.1077-0.3499 under RUN 4 against
#          0.0000-0.0020 under RUN 3, and one live frame reached 0.67, over the
#          0.5 threshold. A clean class that carries logos is harder to separate,
#          the model grew less certain (temperature 1.08 -> 2.12), and the margin
#          went with it. RUN 5 keeps the logos and buys the margin back with more
#          camera-formed training data and more epochs.
# RUN 6 replaces camera simulation as the deployment evidence. Synthetic
# captures remain augmentation, but exact post-crop PNGs dumped by the app
# are mandatory and are group-split by payload hash before this notebook.
RUN = 'run6_real_camera'
OUT = f'{BASE}/{RUN}'

DATA = '/content/structural_data'   # local scratch (regenerated cheaply)
for d in [BASE, OUT, f'{OUT}/eval', f'{OUT}/artifacts']:
    os.makedirs(d, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
IMG_SIZE = 224
CLASS_NAMES = ['clean', 'adversarial', 'tampered']

# Fraction of images in EVERY class rendered as if photographed. It has to apply
# to all three classes: if only the attacks were softened the model would simply
# relearn "soft = attacked", which is the bug being fixed.
#
# RUN 5 raises this from 0.5. Every scan the app will ever make is camera-formed,
# so half the training set being pristine renders spends capacity on a case that
# only arises in testing. The pristine half is still needed - it is what keeps a
# straight PNG upload working, and /scan accepts those - so this is a shift in
# emphasis rather than a removal.
CAMERA_FRACTION = 0.7

# Fail loudly rather than 40 minutes later in Phase 2.
import requests as _rq
print('requests', _rq.__version__, '(google.colab needs 2.32.x - a 2.25.x here means '
      'torchattacks was installed with its deps)')
import torchattacks as _ta  # noqa: F401  - proves --no-deps still gives a usable package

print('Device:', DEVICE, '| GPU:', torch.cuda.get_device_name(0) if DEVICE=='cuda' else 'none')
if DEVICE != 'cuda':
    print('WARNING: enable T4 before Phase 3.')
print('Run:', RUN, '| Out:', OUT)

## Phase 1 - Clean QR base set

2500 codes generated with the `qrcode` library, across all four error-correction
levels and a range of module and border sizes.

**Changed in RUN 4.** Plain black-on-white codes are not what the class looks
like in the wild, and training on only those made `pristine clean recall` a
1.0000 on a question nobody finds hard. Clean codes now also vary in colour and
quiet-zone width, and roughly half of the high-error-correction ones carry a
**centred logo**, as branded codes do everywhere.

That last one is the important one. A logo is a coloured patch sitting on top of
a code - which is also a description of the sticker attack in Phase 2. The
difference is that a logo is centred, small and ringed in white, while a sticker
is off-centre and unringed. Their size ranges deliberately overlap, so the model
has to learn position and shape rather than "is there a patch". Without this a
deployed system would reject every marketing QR it met.

In [ ]:
# ============ Phase 1 — Build clean QR base ============
import qrcode

N_BASE = 2500
CLEAN_DIR = f'{DATA}/clean'
os.makedirs(CLEAN_DIR, exist_ok=True)
USE_FIGSHARE = False
FIGSHARE_DIR = '/content/drive/MyDrive/FYP2/data/clean_qrs'

done_marker = f'{BASE}/clean_manifest.json'
if os.path.exists(done_marker) and os.path.exists(CLEAN_DIR) and len(os.listdir(CLEAN_DIR)) > 100:
    clean_files = json.load(open(done_marker))
    print(f'Phase 1 already done -- {len(clean_files)} clean QRs.')
elif USE_FIGSHARE and os.path.isdir(FIGSHARE_DIR):
    import shutil
    clean_files = []
    for i, f in enumerate(sorted(os.listdir(FIGSHARE_DIR))[:N_BASE]):
        dst = f'{CLEAN_DIR}/clean_{i:05d}.png'
        img = Image.open(f'{FIGSHARE_DIR}/{f}').convert('RGB').resize((IMG_SIZE, IMG_SIZE))
        img.save(dst); clean_files.append(dst)
    json.dump(clean_files, open(done_marker, 'w'))
    print(f'Loaded {len(clean_files)} clean QRs from Figshare.')
else:
    rng = random.Random(SEED)

    def add_logo(img, rng):
        """Stamp a centred brand mark, the way a real branded QR carries one.

        Kept centred, small, and ringed in white - that is what the error
        correction tolerates and what a designer actually does. The tampering
        stickers in Phase 2 are off-centre, unringed rectangles. Size ranges
        overlap on purpose: position and shape are the real distinction, and
        forcing the model to use them instead of "is there a coloured patch" is
        the entire point of this class.
        """
        from PIL import ImageDraw
        W, H = img.size
        side = int(W * rng.uniform(0.10, 0.16))
        ring = max(2, side // 8)
        cx, cy = W // 2, H // 2
        d = ImageDraw.Draw(img)
        box = [cx - side//2 - ring, cy - side//2 - ring,
               cx + side//2 + ring, cy + side//2 + ring]
        d.rectangle(box, fill='white')
        colour = rng.choice([(20,20,20), (200,30,40), (30,90,200), (20,140,80), (240,140,20)])
        inner = [cx - side//2, cy - side//2, cx + side//2, cy + side//2]
        shape = rng.choice(['circle', 'rounded', 'square'])
        if shape == 'circle':
            d.ellipse(inner, fill=colour)
        elif shape == 'rounded':
            d.rounded_rectangle(inner, radius=side//4, fill=colour)
        else:
            d.rectangle(inner, fill=colour)
        return img

    def rand_content(i):
        kind = i % 4
        tok = ''.join(rng.choice('abcdefghijklmnopqrstuvwxyz0123456789') for _ in range(rng.randint(6, 24)))
        if kind == 0: return f'https://{tok}.com/{rng.randint(1,9999)}'
        if kind == 1: return f'https://sub.{tok}.org/path/page?id={rng.randint(1,9999)}'
        if kind == 2: return f'WIFI:T:WPA;S:{tok};P:{tok[:8]};;'
        return f'{tok}-{rng.randint(100000,999999)}'
    clean_files = []
    for i in range(N_BASE):
        ec = rng.choice([qrcode.constants.ERROR_CORRECT_L, qrcode.constants.ERROR_CORRECT_M,
                         qrcode.constants.ERROR_CORRECT_Q, qrcode.constants.ERROR_CORRECT_H])
        qr = qrcode.QRCode(version=None, error_correction=ec,
                           box_size=rng.randint(6, 12), border=rng.randint(1, 6))
        qr.add_data(rand_content(i)); qr.make(fit=True)
        # Real codes are not all black on white. Keep the contrast high enough to
        # stay scannable - a washed-out pair would be an unreadable code, which is
        # a different problem from the one being studied.
        if rng.random() < 0.35:
            dark = tuple(rng.randint(0, 90) for _ in range(3))
            light = tuple(rng.randint(200, 255) for _ in range(3))
        else:
            dark, light = (0, 0, 0), (255, 255, 255)
        img = qr.make_image(fill_color=dark, back_color=light).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
        # Error correction H tolerates roughly 30% loss, so a small centred logo
        # leaves the code readable. About half of real branded codes have one.
        if ec == qrcode.constants.ERROR_CORRECT_H and rng.random() < 0.55:
            img = add_logo(img, rng)
        elif rng.random() < 0.20:
            img = add_logo(img, rng)
        dst = f'{CLEAN_DIR}/clean_{i:05d}.png'; img.save(dst); clean_files.append(dst)
        if (i+1) % 500 == 0: print(f'  generated {i+1}/{N_BASE}')
    json.dump(clean_files, open(done_marker, 'w'))
    print(f'Phase 1 complete -- generated {len(clean_files)} clean QRs.')


## Phase 2 - Split first, then derive adversarial + tampered

Split the clean base BEFORE deriving anything, so no base QR appears in two splits.

**Changed in RUN 2.** `blur` is no longer one of the tampering operations. Blur
describes how a code was *captured*, not how it was *attacked*, and including it
taught RUN 1 to reject every real photograph. The attacks are now sticker
overlay, occlusion, finder-pattern damage and scratches - all of which survive
being photographed and all of which are things an attacker physically does.

**Changed in RUN 3.** `simulate_capture` now starts by compositing the code onto
a procedurally generated surface (wall, wood, paper, gradient, concrete) at 55-90%
of the frame, so there is real background around it. RUN 2 warped the code alone
and replicated its edge pixels, and that omission was the entire remaining gap:
a real camera crop scored 0.5973 while RUN 2's own simulated captures scored
0.06, and compositing onto room texture reproduced the real number (0.6089)
almost exactly.

Half of the images in **every** class go through this, so no capture cue can
identify a class, and the test set contains camera-like samples too.

In [ ]:
# ============ Phase 2 - Adversarial + synthetic tampering, per split ============
import torchattacks
from torchvision import models, transforms
import cv2

to_tensor = transforms.ToTensor()
nprng = np.random.RandomState(SEED)   # separate stream so image noise is reproducible

# ---- split base clean ----
random.Random(SEED).shuffle(clean_files)
n = len(clean_files); n_tr = int(0.70*n); n_va = int(0.15*n)
split_of = {}
for f in clean_files[:n_tr]: split_of[f] = 'train'
for f in clean_files[n_tr:n_tr+n_va]: split_of[f] = 'val'
for f in clean_files[n_tr+n_va:]: split_of[f] = 'test'

# ---- adversarial generator (FYP1-style) ----
victim = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1).eval().to(DEVICE)
class NormModel(nn.Module):
    def __init__(s, m, mean, std):
        super().__init__(); s.m = m
        s.register_buffer('mean', torch.tensor(mean).view(1,3,1,1))
        s.register_buffer('std', torch.tensor(std).view(1,3,1,1))
    def forward(s, x): return s.m((x - s.mean) / s.std)
norm_victim = NormModel(victim, IMAGENET_MEAN, IMAGENET_STD).to(DEVICE).eval()

def make_adversarial(img):
    x = to_tensor(img.resize((IMG_SIZE, IMG_SIZE)).convert('RGB')).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        y = norm_victim(x).argmax(1)
    # Widened downwards: a small-eps perturbation is a genuinely hard case, and
    # excluding it was part of why adversarial recall read 1.0000 on pristine.
    eps = random.uniform(0.004, 0.05)
    atk = (torchattacks.FGSM(norm_victim, eps=eps) if random.random() < 0.5
           else torchattacks.PGD(norm_victim, eps=eps, alpha=0.005, steps=40))
    adv = atk(x, y)[0].detach().cpu().numpy()
    return Image.fromarray((np.clip(adv,0,1)*255).astype(np.uint8).transpose(1,2,0))

# ---- synthetic tampering (Option B): visible, realistic QR attacks ----
# RUN 2: 'blur' removed. It is a property of the photograph, not of the attack,
# and it was the single feature RUN 1 keyed on.
def make_tampered(img):
    im = np.array(img.resize((IMG_SIZE, IMG_SIZE)).convert('RGB'))
    H, W = im.shape[:2]
    ops = random.sample(['sticker', 'occlude', 'finder', 'scratch'],
                        k=random.randint(1, 2))
    for op in ops:
        if op == 'sticker':                       # opaque colored rectangle overlay
            # RUN 4 spreads the size instead of always covering a sixth of the
            # code. The small end sits near the detection limit, which is where
            # a recall number stops being decorative and starts being a claim.
            w = random.randint(W//12, W//3); h = random.randint(H//12, H//3)
            x, y = random.randint(0, W-w), random.randint(0, H-h)
            color = random.choice([(255,255,255),(0,0,0),(220,40,40),(40,120,220)])
            im[y:y+h, x:x+w] = color
        elif op == 'occlude':                     # partial black/white cover
            w, h = random.randint(W//10, W//2), random.randint(H//12, H//4)
            x, y = random.randint(0, W-w), random.randint(0, H-h)
            im[y:y+h, x:x+w] = random.choice([0, 255])
        elif op == 'finder':                      # corrupt one corner finder pattern
            corners = [(0,0),(0,W-W//4),(H-H//4,0)]
            cy, cx = random.choice(corners); s = W//4
            patch = im[cy:cy+s, cx:cx+s]
            patch[:] = nprng.randint(0, 256, patch.shape).astype(np.uint8)
        elif op == 'scratch':                     # random light/dark lines
            for _ in range(random.randint(1, 5)):
                p1 = (random.randint(0,W), random.randint(0,H))
                p2 = (random.randint(0,W), random.randint(0,H))
                cv2.line(im, p1, p2, random.choice([0,255]), random.randint(1,5))
    return Image.fromarray(im)

# ---- camera simulation: what a phone actually hands to the model ----
def random_background(rng, size):
    """A surface a code might be stuck on.

    Generated procedurally rather than sampled from a texture dataset, so the
    notebook stays self-contained - nothing to download, nothing to cite, and
    the backgrounds cannot leak between splits.
    """
    H = W = size
    kind = rng.choice(['wall', 'wood', 'paper', 'gradient', 'concrete'])
    bg = np.zeros((H, W, 3), np.float32)

    if kind == 'wall':
        bg[:] = [rng.uniform(150, 238) for _ in range(3)]
        bg += cv2.GaussianBlur(nprng.normal(0, 14, (H, W, 3)).astype(np.float32), (0, 0), 9)
    elif kind == 'wood':
        bg[:] = [rng.uniform(120, 195), rng.uniform(85, 155), rng.uniform(55, 115)]
        grain = (np.sin(np.linspace(0, rng.uniform(8, 26), H) + rng.uniform(0, 6))
                 * rng.uniform(6, 20))
        bg += grain[:, None, None]
        bg += cv2.GaussianBlur(nprng.normal(0, 9, (H, W, 3)).astype(np.float32), (0, 0), 3)
    elif kind == 'paper':
        bg[:] = [rng.uniform(205, 250) for _ in range(3)]
        bg += nprng.normal(0, 5, (H, W, 3)).astype(np.float32)
    elif kind == 'gradient':
        c0 = np.array([rng.uniform(60, 240) for _ in range(3)], np.float32)
        c1 = np.array([rng.uniform(60, 240) for _ in range(3)], np.float32)
        # Tile to the full frame: broadcasting a (1,W,1) ramp against a (3,)
        # colour silently yields a 1-pixel-tall image, and the paste below then
        # fails on an empty slice.
        if rng.random() < 0.5:
            t = np.tile(np.linspace(0, 1, W, dtype=np.float32)[None, :, None], (H, 1, 1))
        else:
            t = np.tile(np.linspace(0, 1, H, dtype=np.float32)[:, None, None], (1, W, 1))
        bg = c0 * (1 - t) + c1 * t
    else:  # concrete: a couple of blurred noise octaves
        for octave, sigma in ((1, 21), (0.5, 7), (0.25, 3)):
            bg += octave * cv2.GaussianBlur(
                nprng.normal(0, 40, (H, W, 3)).astype(np.float32), (0, 0), sigma)
        bg += rng.uniform(110, 200)

    # Clutter. Smooth surfaces alone turned out to be far too gentle: measured
    # against RUN 2, purely smooth backgrounds left a safe code at mean 0.24
    # while the real room - wood grain, furniture edges, a poster border -
    # reached 0.61. Real scenes have hard edges in them, so add some.
    for _ in range(rng.randint(1, 4)):
        colour = tuple(float(rng.uniform(0, 255)) for _ in range(3))
        if rng.random() < 0.5:
            x1, y1 = rng.randint(-W//3, W), rng.randint(-H//3, H)
            cv2.rectangle(bg, (x1, y1),
                          (x1 + rng.randint(W//6, W), y1 + rng.randint(H//6, H)),
                          colour, -1)
        else:
            cv2.line(bg, (rng.randint(0, W), rng.randint(0, H)),
                     (rng.randint(0, W), rng.randint(0, H)),
                     colour, rng.randint(2, 12))
    bg = cv2.GaussianBlur(bg, (0, 0), rng.uniform(0.5, 2.5))
    return np.clip(bg, 0, 255)


def simulate_capture(img, rng=random):
    """Turn a pristine render into something that looks photographed.

    Applied to every class, so none of these cues can identify a class. Steps 3
    and 4 are what RUN 1 keyed on; step 0 is what RUN 2 was missing.
    """
    im = np.array(img.convert('RGB')).astype(np.float32)
    H, W = im.shape[:2]

    # 0. Put the code ON something. A phone never frames a code edge to edge -
    #    there is always wall, table or paper around it. RUN 2 replicated the
    #    code's own edge pixels instead, and that single omission left a real
    #    camera crop at 0.5973 while its simulated equivalents sat at 0.06.
    scale = rng.uniform(0.55, 0.90)
    ih = max(8, int(H * scale)); iw = max(8, int(W * scale))
    inner = cv2.resize(im, (iw, ih), interpolation=cv2.INTER_AREA)
    canvas = random_background(rng, W)
    oy = rng.randint(0, H - ih); ox = rng.randint(0, W - iw)
    # Codes in the wild are usually printed on something - a poster, a label, a
    # receipt - which sits on the surface as a bright rectangle with a hard
    # border. That was literally the case in the capture that exposed this gap.
    if rng.random() < 0.6:
        pad = rng.randint(2, max(3, min(oy, ox, H-oy-ih, W-ox-iw) + 1))
        sheet = float(rng.uniform(200, 255))
        canvas[max(0,oy-pad):oy+ih+pad, max(0,ox-pad):ox+iw+pad] = sheet
    canvas[oy:oy+ih, ox:ox+iw] = inner
    im = canvas

    # 1. Perspective. A code is almost never square-on to the lens.
    m = rng.uniform(0.02, 0.12)
    off = lambda: rng.uniform(-m, m)
    src = np.float32([[0,0],[W,0],[W,H],[0,H]])
    dst = np.float32([[W*off(),      H*off()],
                      [W*(1+off()),  H*off()],
                      [W*(1+off()),  H*(1+off())],
                      [W*off(),      H*(1+off())]])
    im = cv2.warpPerspective(im, cv2.getPerspectiveTransform(src, dst), (W, H),
                             borderMode=cv2.BORDER_REFLECT)

    # 2. Uneven illumination across the code.
    gx, gy = np.meshgrid(np.linspace(-1,1,W), np.linspace(-1,1,H))
    ang = rng.uniform(0, 2*math.pi)
    im *= (1.0 + rng.uniform(0.05, 0.30)*(math.cos(ang)*gx + math.sin(ang)*gy))[..., None]

    # 3. Resolution loss. The sensor, and then the crop, rescale the code.
    f = rng.uniform(0.25, 0.80)
    small = cv2.resize(im, (max(8,int(W*f)), max(8,int(H*f))), interpolation=cv2.INTER_AREA)
    im = cv2.resize(small, (W, H), interpolation=cv2.INTER_LINEAR)

    # 4. Focus and motion softness.
    k = rng.choice([3, 5])
    im = cv2.GaussianBlur(im, (k, k), rng.uniform(0.4, 1.4))

    # 5. Sensor noise, then JPEG - both are part of any phone capture.
    im = np.clip(im + nprng.normal(0, rng.uniform(1.0, 5.0), im.shape), 0, 255).astype(np.uint8)
    ok, enc = cv2.imencode('.jpg', im[:, :, ::-1],
                           [int(cv2.IMWRITE_JPEG_QUALITY), int(rng.uniform(55, 95))])
    if ok:
        im = cv2.imdecode(enc, cv2.IMREAD_COLOR)[:, :, ::-1]
    return Image.fromarray(im)

# ---- build the 3-class dataset on disk ----
manifest = {'train': [], 'val': [], 'test': []}
captured = {}                                   # path -> 1 if camera-simulated
MANI_PATH = f'{OUT}/dataset_manifest.json'
CAPT_PATH = f'{OUT}/capture_flags.json'
_imgs_ok = False
if os.path.exists(MANI_PATH) and os.path.exists(CAPT_PATH):
    _m = json.load(open(MANI_PATH))
    _imgs_ok = all(_m.get(s) and os.path.exists(_m[s][0][0]) for s in ('train','val','test'))
    if _imgs_ok:
        manifest = _m; captured = json.load(open(CAPT_PATH))
if _imgs_ok:
    print('Phase 2 already done (images present):', {s: len(v) for s, v in manifest.items()})
else:
    if os.path.exists(MANI_PATH):
        print('Manifest found but local images were wiped (session reset) -- regenerating deterministically...')
    for cls in CLASS_NAMES:
        for s in manifest: os.makedirs(f'{DATA}/{s}/{cls}', exist_ok=True)
    rng = random.Random(SEED + 7)               # own stream for the capture decision
    t0 = time.time()
    for i, f in enumerate(clean_files):
        s = split_of[f]; base = Image.open(f).convert('RGB')
        variants = [
            (0, 'clean',       base.resize((IMG_SIZE, IMG_SIZE))),
            (1, 'adversarial', make_adversarial(base)),
            (2, 'tampered',    make_tampered(base)),
        ]
        for label, cls, image in variants:
            shot = rng.random() < CAMERA_FRACTION
            if shot:
                image = simulate_capture(image, rng)
            p = f'{DATA}/{s}/{cls}/{i:05d}.png'
            image.save(p)
            manifest[s].append((p, label)); captured[p] = int(shot)
        if (i+1) % 300 == 0: print(f'  {i+1}/{len(clean_files)}  ({time.time()-t0:.0f}s)')
    json.dump(manifest, open(MANI_PATH, 'w'))
    json.dump(captured, open(CAPT_PATH, 'w'))
    for s in manifest: random.Random(SEED).shuffle(manifest[s])
    print('Phase 2 complete:', {s: len(v) for s, v in manifest.items()})
    print('camera-simulated share:',
          round(sum(captured.values())/max(1,len(captured)), 3))

# ---- Exact crops emitted by app/lib/services/qr_cropper.dart ----
# First run ml_training/structural/src/prepare_runtime_captures.py --strict locally, then copy
# the whole runtime_captures directory to this Drive location. The audit
# prevents a synthetic-only run from being exported as camera-ready.
RUNTIME_ROOT = '/content/drive/MyDrive/FYP2/runtime_captures'
RUNTIME_MANIFEST = f'{RUNTIME_ROOT}/manifest.csv'
RUNTIME_AUDIT = f'{RUNTIME_ROOT}/audit.json'
assert os.path.exists(RUNTIME_MANIFEST), (
    'Missing real app crops. Run ml_training/structural/src/prepare_runtime_captures.py '    'data/runtime_captures --strict, then copy that directory to Drive.')
assert os.path.exists(RUNTIME_AUDIT), 'runtime capture audit.json is missing'
runtime_audit = json.load(open(RUNTIME_AUDIT))
assert runtime_audit.get('strict_ready'), (
    'Real-camera capture gate failed: ' + '; '.join(runtime_audit.get('strict_failures', [])))
runtime_rows = list(csv.DictReader(open(RUNTIME_MANIFEST, encoding='utf-8')))
existing = {p for items in manifest.values() for p, _ in items}
runtime_meta = {}
for row in runtime_rows:
    p = f"{RUNTIME_ROOT}/{row['crop_path']}"
    assert os.path.exists(p), f'Missing runtime crop: {p}'
    split, label = row['split'], int(row['label'])
    assert split in manifest and label in (0, 1, 2)
    if p not in existing:
        manifest[split].append((p, label)); existing.add(p)
    captured[p] = 2  # 0 pristine, 1 simulated camera, 2 exact app camera crop
    runtime_meta[p] = {'session_id': row['session_id'], 'group_id': row['group_id']}
json.dump(manifest, open(MANI_PATH, 'w'))
json.dump(captured, open(CAPT_PATH, 'w'))
print('RUN 6 exact camera frames:', len(runtime_rows), '| sessions:',
      runtime_audit['accepted_sessions'])

## Phase 3 - Fine-tune EfficientNet-B0 (3-class)

**Changed in RUN 2.** Training augments on top of the baked-in camera simulation:
random blur, brightness/contrast and perspective. The baked-in share makes the
*test* set realistic; the augmentation makes the *training* signal robust.
Validation and test use a clean `eval_tf`, so nothing is measured through an
augmenting transform.

**Changed in RUN 5.** 20 epochs instead of 15. RUN 4's validation accuracy had not
flattened by epoch 15, which is a sign of a harder class needing more passes
rather than a bigger model. The best checkpoint on validation is saved each epoch,
so extra epochs can cost time but not quality.

In [ ]:
# ============ Phase 3 - Train ============
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

BEST = f'{OUT}/best_efficientnet_b0.pth'
# RUN 5: 20 epochs, up from 15. Accuracy was still climbing at the end of RUN 4,
# which is what a harder class looks like when it is short of training rather than
# short of capacity. Best-on-validation is saved every epoch, so the extra epochs
# cost time and cannot cost quality.
EPOCHS, LR, BATCH = 20, 1e-4, 32

# RUN 2: augment so that no amount of softness or tilt identifies a class.
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomApply([transforms.GaussianBlur(3, sigma=(0.1, 1.2))], p=0.5),
    transforms.RandomApply([transforms.ColorJitter(brightness=0.25, contrast=0.25)], p=0.5),
    transforms.RandomPerspective(distortion_scale=0.15, p=0.3, fill=255),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
# Deterministic transform for val/test/export - never augment what you measure.
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class QRDS(Dataset):
    def __init__(s, items, tf): s.items = items; s.tf = tf
    def __len__(s): return len(s.items)
    def __getitem__(s, i):
        p, y = s.items[i]
        return s.tf(Image.open(p).convert('RGB')), y

def build_model():
    m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, 3)
    return m

# A checkpoint is only reusable if it was trained on the data now on disk. Record
# what that was, and compare, rather than trusting the file's existence.
TRAIN_FP = f'{OUT}/train_fingerprint.json'
_fingerprint = {
    'camera_fraction': CAMERA_FRACTION,
    'n_train': len(manifest['train']),
    'n_val': len(manifest['val']),
    'n_test': len(manifest['test']),
    'simulated_camera_frames': sum(captured.get(p, 0) == 1 for p in captured),
    'real_camera_frames': sum(captured.get(p, 0) == 2 for p in captured),
    'runtime_manifest_sha256': hashlib.sha256(open(RUNTIME_MANIFEST, 'rb').read()).hexdigest(),
    'epochs': EPOCHS,
    'img_size': IMG_SIZE,
}
_recorded = json.load(open(TRAIN_FP)) if os.path.exists(TRAIN_FP) else None

if os.path.exists(BEST) and _recorded == _fingerprint:
    print('Phase 3 already done -- model at', BEST)
else:
    if os.path.exists(BEST):
        print('A checkpoint exists but was trained on different data or settings:')
        print('   trained on:', _recorded)
        print('   now       :', _fingerprint)
        print('Retraining. Delete the run folder instead if you meant to keep the old one.')
        os.remove(BEST)
    # Give exact live-camera frames 40% of training draws without copying
    # files. Their count is much smaller than synthetic data, but they are
    # the distribution the deployed model must actually solve.
    n_real = sum(captured.get(p, 0) == 2 for p, _ in manifest['train'])
    n_other = len(manifest['train']) - n_real
    assert n_real and n_other, 'RUN 6 requires both real and synthetic training rows'
    real_weight = (0.40 * n_other) / (0.60 * n_real)
    sample_weights = [real_weight if captured.get(p, 0) == 2 else 1.0
                      for p, _ in manifest['train']]
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)
    tr = DataLoader(QRDS(manifest['train'], train_tf), batch_size=BATCH,
                    sampler=sampler, num_workers=2)
    va = DataLoader(QRDS(manifest['val'], eval_tf), batch_size=64, num_workers=2)
    model = build_model().to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=3)
    crit = nn.CrossEntropyLoss()
    best_acc = 0.0
    for ep in range(EPOCHS):
        model.train()
        for x, y in tr:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad(); loss = crit(model(x), y); loss.backward(); opt.step()
        model.eval(); correct = tot = 0
        with torch.no_grad():
            for x, y in va:
                x = x.to(DEVICE)
                correct += (model(x).argmax(1).cpu() == y).sum().item(); tot += len(y)
        acc = correct/tot; sched.step(1-acc)
        print(f'Epoch {ep+1}/{EPOCHS}  val_acc={acc:.4f}')
        if acc > best_acc:
            best_acc = acc; torch.save(model.state_dict(), BEST)
    json.dump(_fingerprint, open(TRAIN_FP, 'w'))
    print(f'Phase 3 complete. Best val_acc={best_acc:.4f} -> {BEST}')

## Phase 4 - Evaluation (per class + confusion matrix)

**Changed in RUN 2.** Metrics are also reported separately for pristine and
camera-simulated test images. A single average would hide the number that
actually matters - whether a photographed code is still classified correctly -
and would also hide any loss on the adversarial class, whose FGSM/PGD
perturbation is partly destroyed by blur and downsampling. That loss is a real
physical-world effect, so it should be visible rather than averaged away.

In [ ]:
# ============ Phase 4 - Evaluate ============
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay)

model = build_model().to(DEVICE)
model.load_state_dict(torch.load(BEST, map_location=DEVICE)); model.eval()
te = DataLoader(QRDS(manifest['test'], eval_tf), batch_size=64, num_workers=2)

all_logits, all_y = [], []
with torch.no_grad():
    for x, y in te:
        all_logits.append(model(x.to(DEVICE)).cpu()); all_y.append(y)
logits = torch.cat(all_logits); y_true = torch.cat(all_y).numpy()
np.save(f'{OUT}/eval/test_logits.npy', logits.numpy())
np.save(f'{OUT}/eval/test_labels.npy', y_true)
y_pred = logits.argmax(1).numpy()

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))
rep = classification_report(y_true, y_pred, target_names=CLASS_NAMES, output_dict=True)

# --- report pristine, simulated, and exact app-camera distributions separately ---
capture_kind = np.array([captured.get(p, 0) for p, _ in manifest['test']])
is_shot = capture_kind != 0
slices = {}
for name, mask in (('pristine', capture_kind == 0),
                   ('camera_simulated', capture_kind == 1),
                   ('camera_real_app_crop', capture_kind == 2)):
    if mask.sum() == 0:
        continue
    acc = float((y_pred[mask] == y_true[mask]).mean())
    per_class = {CLASS_NAMES[c]: float((y_pred[mask & (y_true == c)] == c).mean())
                 for c in range(3) if (mask & (y_true == c)).sum()}
    slices[name] = {'n': int(mask.sum()), 'accuracy': acc, 'recall_per_class': per_class}
    print(f'\n{name}: n={mask.sum()}  accuracy={acc:.4f}')
    for c, r in per_class.items():
        print(f'    {c:>12} recall {r:.4f}')

# The RUN 1 failure in one number: a clean code that was photographed must NOT
# be called manipulated.
clean_shot = is_shot & (y_true == 0)
if clean_shot.sum():
    fp = float((y_pred[clean_shot] != 0).mean())
    print(f'\nphotographed CLEAN codes wrongly flagged: {fp:.4f}  (RUN 1 was effectively 1.0)')
    slices['clean_photographed_false_positive_rate'] = fp

# Deployment gate: averages over synthetic images cannot authorize a live
# model. These conditions are evaluated only on grouped, exact app crops.
real_mask = capture_kind == 2
REAL_GATE_FAILURES = []
real_recall = {}
for c, name in enumerate(CLASS_NAMES):
    m = real_mask & (y_true == c)
    assert m.sum(), f'No real-camera {name} test frames after strict audit'
    real_recall[name] = float((y_pred[m] == c).mean())
real_clean_fp = 1.0 - real_recall['clean']
if real_clean_fp > 0.05:
    REAL_GATE_FAILURES.append(f'real clean false-positive {real_clean_fp:.4f} > 0.05')
if real_recall['tampered'] < 0.85:
    REAL_GATE_FAILURES.append(f"real tampered recall {real_recall['tampered']:.4f} < 0.85")
if real_recall['adversarial'] < 0.80:
    REAL_GATE_FAILURES.append(f"real adversarial recall {real_recall['adversarial']:.4f} < 0.80")
slices['real_camera_deployment_gate'] = {
    'recall_per_class': real_recall, 'clean_false_positive_rate': real_clean_fp,
    'passed': not REAL_GATE_FAILURES, 'failures': REAL_GATE_FAILURES}
print('\nReal-camera deployment gate:',
      'PASS' if not REAL_GATE_FAILURES else 'FAIL', REAL_GATE_FAILURES)

rep['slices'] = slices
json.dump(rep, open(f'{OUT}/eval/metrics_test.json', 'w'), indent=2)
cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(cmap='Blues', colorbar=False)
plt.title('Structural 3-class - Test Confusion Matrix (RUN 2)')
plt.savefig(f'{OUT}/eval/confusion_matrix.png', dpi=160, bbox_inches='tight'); plt.close()
print('\nPhase 4 complete. Confusion matrix saved.')

## Phase 5 — Temperature calibration
Fits one temperature T on validation logits so the softmax probabilities are honest — the
fusion engine consumes `p_structural = 1 − P(clean)`, so it must be calibrated.

In [ ]:
# ============ Phase 5 - Temperature scaling ============
TEMP_PATH = f'{OUT}/artifacts/temperature.json'
va2 = DataLoader(QRDS(manifest['val'], eval_tf), batch_size=64, num_workers=2)
vl, vy = [], []
with torch.no_grad():
    for x, y in va2:
        vl.append(model(x.to(DEVICE)).cpu()); vy.append(y)
vl = torch.cat(vl); vy = torch.cat(vy)

T = torch.nn.Parameter(torch.ones(1))
opt = torch.optim.LBFGS([T], lr=0.05, max_iter=200)
nll = torch.nn.CrossEntropyLoss()
def closure():
    opt.zero_grad(); loss = nll(vl / T.clamp(min=1e-3), vy); loss.backward(); return loss
opt.step(closure)
temperature = float(T.detach().clamp(min=1e-3))

def ece_binary(p_manip, y_manip, nb=10):
    bins = np.linspace(0,1,nb+1); tot=0.0
    conf = np.maximum(p_manip, 1-p_manip); correct = ((p_manip>=0.5).astype(int)==y_manip)
    for lo,hi in zip(bins[:-1],bins[1:]):
        m=(conf>lo)&(conf<=hi)
        if m.sum(): tot += m.mean()*abs(correct[m].mean()-conf[m].mean())
    return float(tot)

tl = torch.tensor(np.load(f'{OUT}/eval/test_logits.npy')); ty = np.load(f'{OUT}/eval/test_labels.npy')
y_manip = (ty != 0).astype(int)
p_before = 1 - torch.softmax(tl, -1)[:,0].numpy()
p_after  = 1 - torch.softmax(tl/temperature, -1)[:,0].numpy()
eb, ea = ece_binary(p_before, y_manip), ece_binary(p_after, y_manip)
print(f'T={temperature:.4f}  |  manipulated-vs-clean ECE {eb:.4f} -> {ea:.4f}')
if ea > 0.05:
    REAL_GATE_FAILURES.append(f'calibrated structural ECE {ea:.4f} > 0.05')
json.dump({'temperature': temperature, 'ece_before': eb, 'ece_after': ea},
          open(TEMP_PATH, 'w'), indent=2)
print('Phase 5 complete ->', TEMP_PATH)

## Phase 6 — Sanity check on p_structural
Confirms the deployed signal behaves: clean test QRs → low `p_structural`; adversarial and
tampered → high. Prints the mean `p_structural` per true class (clean should be lowest).

In [ ]:
# ============ Phase 6 - p_structural behaviour ============
temperature = json.load(open(TEMP_PATH))['temperature']
tl = torch.tensor(np.load(f'{OUT}/eval/test_logits.npy')); ty = np.load(f'{OUT}/eval/test_labels.npy')
probs = torch.softmax(tl/temperature, -1).numpy()
p_struct = 1 - probs[:,0]
for c, name in enumerate(CLASS_NAMES):
    m = ty == c
    print(f'{name:>12}: mean p_structural = {p_struct[m].mean():.3f}  '
          f'(should be {"LOW" if c==0 else "HIGH"})')

# The check RUN 1 would have failed: a clean code, photographed, must stay LOW.
capture_kind = np.array([captured.get(p, 0) for p, _ in manifest['test']])
is_shot = capture_kind != 0
for label, mask in (('clean, pristine', (ty==0) & (capture_kind==0)),
                    ('clean, simulated camera', (ty==0) & (capture_kind==1)),
                    ('clean, exact app camera', (ty==0) & (capture_kind==2))):
    if mask.sum():
        print(f'{label:>22}: mean p_structural = {p_struct[mask].mean():.3f} '
              f'(n={mask.sum()})')

manip_true = (ty != 0).astype(int)
acc = ((p_struct >= 0.5).astype(int) == manip_true).mean()
print(f'\nManipulated-vs-clean accuracy @0.5: {acc:.4f}')
json.dump({'mean_p_structural_per_class':
           {CLASS_NAMES[c]: float(p_struct[ty==c].mean()) for c in range(3)},
           'mean_p_structural_clean_photographed':
               float(p_struct[(ty==0) & is_shot].mean()) if ((ty==0) & is_shot).sum() else None,
           'manip_vs_clean_acc': float(acc)},
          open(f'{OUT}/eval/sanity.json','w'), indent=2)
print('Phase 6 complete.')

## Phase 7 — ONNX + INT8 + latency + predict function
Exports to ONNX, applies dynamic INT8 quantization (≤2 pp accuracy-drop policy, else FP32),
benchmarks single-image CPU latency, and defines `predict_structural(pil_image)` returning
`p_structural` + predicted type. Download `structural/artifacts/` → repo `training/artifacts/`.

In [ ]:
# ============ Phase 7 — Export, quantize, benchmark ============
import onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType

if REAL_GATE_FAILURES:
    raise RuntimeError('RUN 6 export refused: ' + '; '.join(REAL_GATE_FAILURES))

ART = f'{OUT}/artifacts'
ONNX_FP32 = f'{ART}/structural_fp32.onnx'; ONNX_INT8 = f'{ART}/structural_int8.onnx'

dummy = torch.randn(1,3,IMG_SIZE,IMG_SIZE).to(DEVICE)
model.eval()
torch.onnx.export(model, dummy, ONNX_FP32, opset_version=14, dynamo=False,
                  input_names=['input'], output_names=['logits'],
                  dynamic_axes={'input':{0:'b'}, 'logits':{0:'b'}})
quantize_dynamic(ONNX_FP32, ONNX_INT8, weight_type=QuantType.QInt8)
print('Exported ONNX FP32 + INT8.')

def sess(p):
    so = ort.SessionOptions(); so.intra_op_num_threads = 1
    return ort.InferenceSession(p, so, providers=['CPUExecutionProvider'])
def onnx_pred(s, arr): return s.run(None, {'input': arr})[0]

tf = eval_tf   # never benchmark or export through the augmenting transform
sample = manifest['test'][:2000]
def acc_of(logits_fn):
    correct = 0
    for p, y in sample:
        arr = tf(Image.open(p).convert('RGB')).unsqueeze(0).numpy()
        if logits_fn(arr).argmax() == y: correct += 1
    return correct/len(sample)
s_fp32, s_int8 = sess(ONNX_FP32), sess(ONNX_INT8)
with torch.no_grad():
    a_torch = np.mean([model(tf(Image.open(p).convert('RGB')).unsqueeze(0).to(DEVICE)
                             ).argmax().item()==y for p,y in sample])
a_fp32 = acc_of(lambda a: onnx_pred(s_fp32, a)[0])
a_int8 = acc_of(lambda a: onnx_pred(s_int8, a)[0])
drop = (a_torch - a_int8)*100
print(f'Acc torch {a_torch:.4f} | fp32 {a_fp32:.4f} | int8 {a_int8:.4f} | drop {drop:.2f} pp')
DEPLOY = ONNX_INT8 if drop <= 2.0 else ONNX_FP32
if drop > 2.0: print('INT8 drop > 2pp -> deploying FP32.')
json.dump({'deploy_model': os.path.basename(DEPLOY)}, open(f'{ART}/deploy_choice.json','w'))

bimg = tf(Image.open(sample[0][0]).convert('RGB')).unsqueeze(0).numpy()
def bench(fn, runs=100, warm=10):
    for _ in range(warm): fn()
    t=[];
    for _ in range(runs):
        t0=time.perf_counter(); fn(); t.append((time.perf_counter()-t0)*1000)
    return round(float(np.median(t)),1), round(float(np.percentile(t,95)),1)
lat = {'onnx_fp32': bench(lambda: onnx_pred(s_fp32, bimg)),
       'onnx_int8': bench(lambda: onnx_pred(s_int8, bimg))}
print('Latency median/P95 ms:', lat)

_T = json.load(open(TEMP_PATH))['temperature']; _ds = sess(DEPLOY)
def predict_structural(pil_img):
    arr = tf(pil_img.convert('RGB')).unsqueeze(0).numpy()
    logits = onnx_pred(_ds, arr)[0]
    probs = torch.softmax(torch.tensor(logits)/_T, -1).numpy()
    return {'p_structural': float(1-probs[0]),
            'predicted_type': CLASS_NAMES[int(probs.argmax())],
            'probs': {CLASS_NAMES[i]: float(probs[i]) for i in range(3)}}
print('\npredict_structural() demo (first test image of each class):')
for c in range(3):
    p = next(p for p,y in manifest['test'] if y==c)
    print(f'  true={CLASS_NAMES[c]:>11}  ->', predict_structural(Image.open(p)))

json.dump({'test_metrics': json.load(open(f'{OUT}/eval/metrics_test.json')),
           'calibration': json.load(open(TEMP_PATH)),
           'sanity': json.load(open(f'{OUT}/eval/sanity.json')),
           'quantization': {'acc_torch': float(a_torch), 'acc_int8': float(a_int8),
                            'drop_pp': float(drop), 'deployed': os.path.basename(DEPLOY)},
           'latency_ms': lat}, open(f'{ART}/metrics_summary.json','w'), indent=2)
print('\nPhase 7 complete. Download', ART, '-> QRGuard/training/artifacts/structural/')

## Done — checklist
Download `MyDrive/FYP2/structural/artifacts/` → repo `training/artifacts/structural/`:
`structural_int8.onnx` (or fp32 per `deploy_choice.json`), `temperature.json`,
`metrics_summary.json`. Keep `eval/confusion_matrix.png` and record the structural results for the FYP report.